# Resume Analyzer vs. Job Description

## What this does:
1. Parses your resume PDF
2. Scrapes the job description from a URL
3. Uses GPT to score the match, identify gaps, and rewrite weak bullets

**Based on the same pattern as the Day 5 Brochure project.**

In [ ]:
# Imports
import os
import sys
import json
import pdfplumber
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# scraper.py lives one level up (week1/) — add it to path
sys.path.insert(0, '..')
from scraper import fetch_website_contents

In [2]:
# Initialize
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-') and len(api_key) > 10:
    print("API key looks good")
else:
    print("Check your API key in .env")

MODEL = 'gpt-4.1-mini'
openai = OpenAI()

API key looks good


## Step 1 — Set your inputs here

In [3]:
# ---- CHANGE THESE TWO LINES ----
RESUME_PATH = "/Users/karthikpinneboyina/Desktop/Karthik_Pinneboyina-Resume.pdf"          # Path to your resume PDF (relative to this notebook)
JOB_URL     = "https://careers.wipro.com/job/DEVELOPER-L3/158313-en_US"  # The job posting URL
# --------------------------------

## Step 2 — Parse the resume PDF

In [4]:
def extract_resume_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

resume_text = extract_resume_text(RESUME_PATH)
print(f"Extracted {len(resume_text)} characters from resume")
print("--- Preview (first 500 chars) ---")
print(resume_text[:500])

Extracted 3712 characters from resume
--- Preview (first 500 chars) ---
Karthik Pinneboyina
Hyderabad, Telangana | +91 7997758461 | karthikpinneboyina99@gmail.com | LinkedIn | GitHub
TECHNICAL SKILLS
Languages: Python, TypeScript, JavaScript
Frontend: React, Next.js, Tailwind CSS, shadcn/ui, Recharts, Framer Motion
Backend: FastAPI, Express, REST APIs
Database: PostgreSQL, SQLite
AI / GenAI: Prompt Engineering, Multi-Agent LLM Systems, LLM Integration (OpenRouter, Cerebras API), XGBoost, OpenCV,
KNN, Machine Learning
Tools & Practices: Git, Docker, GitHub Actions (C


## Step 3 — Get the job description

**Option A (recommended):** Paste the job description text directly — works for LinkedIn, Indeed, Glassdoor, any site.

**Option B:** Scrape from URL — only works for simple company career pages (not LinkedIn/Indeed).

If Option B gives you cookie/login text, use Option A.

In [6]:
# OPTION A: Paste the job description here (copy from browser, paste between the triple quotes)
# Leave the placeholder text if you want to use Option B (URL scraping) below.

JOB_DESCRIPTION_TEXT = """
Job Description
Role Purpose

The purpose of this role is to design, test and maintain software programs for operating systems or applications which needs to be deployed at a client end and ensure its meet 100% quality assurance parameters

Do

1. Instrumental in understanding the requirements and design of the product/ software

Develop software solutions by studying information needs, studying systems flow, data usage and work processes
Investigating problem areas followed by the software development life cycle
Facilitate root cause analysis of the system issues and problem statement
Identify ideas to improve system performance and impact availability
Analyze client requirements and convert requirements to feasible design
Collaborate with functional teams or systems analysts who carry out the detailed investigation into software requirements
Conferring with project managers to obtain information on software capabilities

2. Perform coding and ensure optimal software/ module development

Determine operational feasibility by evaluating analysis, problem definition, requirements, software development and proposed software
Develop and automate processes for software validation by setting up and designing test cases/scenarios/usage cases, and executing these cases
Modifying software to fix errors, adapt it to new hardware, improve its performance, or upgrade interfaces.
Analyzing information to recommend and plan the installation of new systems or modifications of an existing system
Ensuring that code is error free or has no bugs and test failure
Preparing reports on programming project specifications, activities and status
Ensure all the codes are raised as per the norm defined for project / program / account with clear description and replication patterns
Compile timely, comprehensive and accurate documentation and reports as requested
Coordinating with the team on daily project status and progress and documenting it
Providing feedback on usability and serviceability, trace the result to quality risk and report it to concerned stakeholders

3. Status Reporting and Customer Focus on an ongoing basis with respect to project and its execution

Capturing all the requirements and clarifications from the client for better quality work
Taking feedback on the regular basis to ensure smooth and on time delivery
Participating in continuing education and training to remain current on best practices, learn new programming languages, and better assist other team members.
Consulting with engineering staff to evaluate software-hardware interfaces and develop specifications and performance requirements
Document and demonstrate solutions by developing documentation, flowcharts, layouts, diagrams, charts, code comments and clear code
Documenting very necessary details and reports in a formal way for proper understanding of software from client proposal to implementation
Ensure good quality of interaction with customer w.r.t. e-mail content, fault report tracking, voice calls, business etiquette etc
Timely Response to customer requests and no instances of complaints either internally or externally
"""

In [7]:
COOKIE_WALL_SIGNALS = ["cookie", "consent", "log in", "sign in", "enable javascript"]

def is_cookie_wall(text):
    text_lower = text.lower()
    return any(signal in text_lower for signal in COOKIE_WALL_SIGNALS) and len(text) < 2000

# Auto-pick: use pasted text if provided, otherwise try scraping
if JOB_DESCRIPTION_TEXT.strip() and JOB_DESCRIPTION_TEXT.strip() != "PASTE THE FULL JOB DESCRIPTION HERE":
    jd_text = JOB_DESCRIPTION_TEXT.strip()
    print(f"Using pasted job description ({len(jd_text)} characters)")
else:
    print(f"Scraping job description from {JOB_URL}...")
    jd_text = fetch_website_contents(JOB_URL)
    if is_cookie_wall(jd_text):
        print("\nGot a cookie/login wall instead of job content.")
        print("Go back and paste the job description into the Option A cell above.")
        jd_text = ""
    else:
        print(f"Scraped {len(jd_text)} characters successfully")

if jd_text:
    print("--- Preview (first 500 chars) ---")
    print(jd_text[:500])

Using pasted job description (3134 characters)
--- Preview (first 500 chars) ---
Job Description
Role Purpose

The purpose of this role is to design, test and maintain software programs for operating systems or applications which needs to be deployed at a client end and ensure its meet 100% quality assurance parameters

Do

1. Instrumental in understanding the requirements and design of the product/ software

Develop software solutions by studying information needs, studying systems flow, data usage and work processes
Investigating problem areas followed by the software deve


## Step 4 — LLM Call 1: Match Score

In [8]:
score_system_prompt = """
You are an expert recruiter and resume coach. You compare a candidate's resume against a job description
and score how well they match. Be honest and precise.

Respond in JSON with this exact structure:
{
    "overall_score": <integer 0-100>,
    "grade": <"A" | "B" | "C" | "D" | "F">,
    "section_scores": {
        "skills_match": <integer 0-100>,
        "experience_relevance": <integer 0-100>,
        "education_fit": <integer 0-100>,
        "keywords_coverage": <integer 0-100>
    },
    "summary": "<2-3 sentence honest summary of the match>"
}
"""

def get_match_score(resume_text, jd_text):
    user_prompt = f"""
## Resume:
{resume_text[:4000]}

## Job Description:
{jd_text[:3000]}

Score how well this resume matches the job description.
"""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": score_system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("Scoring resume match...")
score_result = get_match_score(resume_text, jd_text)
print(json.dumps(score_result, indent=2))

Scoring resume match...
{
  "overall_score": 72,
  "grade": "B",
  "section_scores": {
    "skills_match": 80,
    "experience_relevance": 65,
    "education_fit": 85,
    "keywords_coverage": 65
  },
  "summary": "The candidate has strong technical skills relevant to software development including programming languages, frontend and backend frameworks, and testing frameworks which align well with the job requirements. Their hands-on project experience demonstrates practical software design, development, and deployment capabilities. However, the resume lacks explicit emphasis on some key job responsibilities such as formal QA process adherence, client communication, and detailed documentation practices, which lowers experience relevance and keywords coverage slightly. Education is a good fit with an MS in Computer Science."
}


## Step 5 — LLM Call 2: Gap Analysis

In [9]:
gap_system_prompt = """
You are an expert recruiter. Identify what is missing from the candidate's resume compared to the job description.
Be specific and actionable — focus on things the candidate could realistically add or highlight.

Respond in JSON with this exact structure:
{
    "missing_hard_skills": ["skill1", "skill2", ...],
    "missing_soft_skills": ["skill1", ...],
    "missing_keywords": ["keyword1", ...],
    "experience_gaps": ["gap description1", ...],
    "quick_wins": ["action candidate can take1", ...]
}
"""

def get_gap_analysis(resume_text, jd_text):
    user_prompt = f"""
## Resume:
{resume_text[:4000]}

## Job Description:
{jd_text[:3000]}

Identify what is missing from this resume for this job.
"""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": gap_system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("Analyzing gaps...")
gap_result = get_gap_analysis(resume_text, jd_text)
print(json.dumps(gap_result, indent=2))

Analyzing gaps...
{
  "missing_hard_skills": [
    "Operating Systems Software Development",
    "Software Validation Automation (Test Case/Scenario Design & Execution)",
    "Bug Tracking and Debugging Tools",
    "Installation and Configuration of New Systems",
    "Performance Tuning and Optimization for Client-Deployed Software",
    "Formal Documentation Standards (Flowcharts, Layouts, Diagrams, Charts)",
    "Client Requirement Gathering and Clarification Processes",
    "Software Development Life Cycle (SDLC) in Client-Facing Deployments"
  ],
  "missing_soft_skills": [
    "Client Communication and Feedback Management",
    "Status Reporting and Project Progress Coordination",
    "Collaborating with Project Managers and Functional Teams",
    "Customer Focus and Relationship Management",
    "Participating in Continuing Education and Training"
  ],
  "missing_keywords": [
    "Quality Assurance Parameters",
    "Root Cause Analysis",
    "System Issue Investigation",
    "Oper

## Step 6 — LLM Call 3: Bullet Point Rewrites

In [10]:
bullet_system_prompt = """
You are a professional resume writer. Rewrite the candidate's resume bullet points to better match
the job description. Use strong action verbs, quantify impact where possible, and incorporate
relevant keywords from the job description naturally.

Respond in JSON with this exact structure:
{
    "rewrites": [
        {
            "original": "<original bullet>",
            "improved": "<rewritten bullet>",
            "why": "<one sentence explaining the improvement>"
        }
    ],
    "new_bullets_to_add": ["<new bullet that highlights something missing but implied by experience>"]
}

Focus on the 4-6 most impactful rewrites. Do not invent experience that is not in the resume.
"""

def get_bullet_rewrites(resume_text, jd_text):
    user_prompt = f"""
## Resume:
{resume_text[:4000]}

## Job Description:
{jd_text[:3000]}

Rewrite the resume bullet points to better match this job description.
"""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": bullet_system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("Generating bullet rewrites...")
bullet_result = get_bullet_rewrites(resume_text, jd_text)
print(json.dumps(bullet_result, indent=2))

Generating bullet rewrites...
{
  "rewrites": [
    {
      "original": "Engineered and refined prompts for LLM-powered automation workflows, improving output accuracy and consistency across use cases.",
      "improved": "Designed and optimized AI-driven automation workflows by refining prompt engineering techniques, enhancing system accuracy and reliability in alignment with operational requirements.",
      "why": "This rewrite emphasizes the candidate's role in designing solutions and improving system performance, aligning with the job's focus on software design and quality assurance."
    },
    {
      "original": "Built AI automation tools integrating LLM APIs to streamline repetitive operational tasks and cut manual processing effort.",
      "improved": "Developed and deployed AI automation software by integrating LLM APIs to increase operational efficiency and reduce manual process time, ensuring seamless client-end deployment.",
      "why": "Highlights development and deplo

## Step 7 — Generate the Full Report (streamed)

In [11]:
report_system_prompt = """
You are a career coach producing a clear, actionable resume analysis report.
Write in markdown. Be honest but encouraging. Structure the report with clear sections.
Use tables, bullet points, and bold text to make it scannable.
"""

def stream_full_report(score_result, gap_result, bullet_result, resume_text, jd_text):
    user_prompt = f"""
Here is the analysis data for a resume vs. job description match. 
Write a professional, actionable report using this data.

## Match Score Data:
{json.dumps(score_result, indent=2)}

## Gap Analysis Data:
{json.dumps(gap_result, indent=2)}

## Bullet Rewrite Suggestions:
{json.dumps(bullet_result, indent=2)}

Structure the report with these sections:
1. Overall Match Score (show the score visually with a progress bar using unicode blocks)
2. Score Breakdown (table of section scores)
3. Summary Assessment
4. Skills & Keyword Gaps
5. Resume Bullet Rewrites (show original → improved side by side)
6. New Bullets to Add
7. Quick Wins (top 3 things to do right now)
"""
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": report_system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

print("Generating full report...")
stream_full_report(score_result, gap_result, bullet_result, resume_text, jd_text)

Generating full report...


# Resume Analysis Report

---

## 1. Overall Match Score

**Score:** 72 / 100  
**Grade:** B  

Progress:  
```
████████████████████████████████████░░░░░░░░░░ 72%
```

Your resume shows a solid fit for the role, reflecting strong technical skills and relevant education. There is opportunity to enhance relevance and keyword alignment further.

---

## 2. Score Breakdown

| Section                | Score (%) | Notes                                           |
|------------------------|-----------|------------------------------------------------|
| **Skills Match**       | 80        | Strong technical proficiency aligns well       |
| **Experience Relevance** | 65        | Needs clearer emphasis on key job responsibilities |
| **Education Fit**      | 85        | Good alignment with an MS in Computer Science  |
| **Keywords Coverage**  | 65        | Some important job keywords are missing        |

---

## 3. Summary Assessment

You demonstrate **strong technical skills** in software development — including programming languages, frontend/backend frameworks, and testing tools — that closely match the job needs. Your hands-on project work shows practical ability in design and deployment.

However, the resume falls short in emphasizing some critical job functions such as:

- Formal adherence to QA processes  
- Client communications and feedback handling  
- Detailed documentation and reporting practices  

These gaps are reflected in somewhat lower experience relevance and keyword coverage scores.

Your educational background is well-aligned, strengthening your overall candidacy.

---

## 4. Skills & Keyword Gaps

### Missing Hard Skills

- Operating Systems Software Development  
- Software Validation Automation (Test Case/Scenario Design & Execution)  
- Bug Tracking and Debugging Tools  
- Installation and Configuration of New Systems  
- Performance Tuning and Optimization for Client-Deployed Software  
- Formal Documentation Standards (Flowcharts, Layouts, Diagrams, Charts)  
- Client Requirement Gathering and Clarification Processes  
- Software Development Life Cycle (SDLC) in Client-Facing Deployments  

### Missing Soft Skills

- Client Communication and Feedback Management  
- Status Reporting and Project Progress Coordination  
- Collaborating with Project Managers and Functional Teams  
- Customer Focus and Relationship Management  
- Participating in Continuing Education and Training  

### Missing Keywords

- Quality Assurance Parameters  
- Root Cause Analysis  
- System Issue Investigation  
- Operational Feasibility Evaluation  
- Test Failure Analysis  
- Coding Norms and Standards  
- Traceability to Quality Risk  
- Fault Report Tracking  

### Experience Gaps

- Lack of explicit client requirements analysis and design conversion experience.  
- No evidence of formal root cause analysis or problem investigation in projects.  
- Absence of documented software validation automation including test case design and execution.  
- Missing formal reporting, documentation, and status update involvement.  
- No clear experience delivering client-deployed OS/application software with QA standards.  
- Missing collaboration examples with project management or functional teams for progress/requirements.  

---

## 5. Resume Bullet Rewrites

| **Original** | **Improved** |
|--------------|--------------|
| Engineered and refined prompts for LLM-powered automation workflows, improving output accuracy and consistency across use cases. | Designed and optimized AI-driven automation workflows by refining prompt engineering techniques, enhancing system accuracy and reliability in alignment with operational requirements. |
| Built AI automation tools integrating LLM APIs to streamline repetitive operational tasks and cut manual processing effort. | Developed and deployed AI automation software by integrating LLM APIs to increase operational efficiency and reduce manual process time, ensuring seamless client-end deployment. |
| Tested and iterated on prompt templates across model versions, maintaining a versioned prompt library for reuse across projects. | Conducted thorough testing and iterative improvement of prompt templates, establishing a version-controlled library to maintain code quality and facilitate reuse, adhering to software validation best practices. |
| Collaborated with engineering teams to integrate AI automation tooling into existing product workflows supporting enterprise clients. | Partnered with cross-functional engineering teams to integrate AI automation tools into production workflows, ensuring system compatibility and meeting enterprise client requirements. |
| Built a full-stack AI project-management platform from scratch with a drag-and-drop Kanban board (@dnd-kit), role-based auth for Admin/Employee/External users (JWT + bcrypt), and analytics dashboards (Recharts). | Architected and implemented a full-stack AI-driven project management platform featuring modular design, role-based authentication (JWT, bcrypt), and interactive analytics dashboards, ensuring high usability and system robustness. |
| Engineered a win-probability prediction engine with a 6-axis team-strength radar (Recharts) and key-edge factor analysis across major football & cricket tournaments. | Developed a predictive analytics engine with multi-dimensional data visualization to provide actionable insights, enhancing system performance and decision-making capabilities in sports analytics applications. |

---

## 6. New Bullets to Add

- Documented comprehensive software design, test cases, and user manuals to ensure clear communication with stakeholders and facilitate seamless client deployment.  
- Conducted root cause analysis of identified system issues and implemented corrective software modifications to improve stability and performance.  

---

## 7. Quick Wins (Top 3 Actions Now)

1. **Incorporate key missing keywords** such as "quality assurance," "root cause analysis," "test failure," and "coding standards" into bullet points and summary sections to improve ATS and human reader alignment.  
2. **Add concrete examples or project details** showing end-to-end software development targeting client deployments with explicit QA and validation focus.  
3. **Highlight communication and collaboration skills** by mentioning client interactions, requirement gathering, feedback management, and cross-team coordination experience.  

---

### Final Encouragement

Your resume already communicates strong technical expertise and education — a great foundation! By weaving in the targeted responsibilities, soft skills, and keywords detailed above, you'll present a more comprehensive, role-aligned profile that significantly improves your competitiveness for this position.

Keep up the good work!

## One-shot: Run everything at once

In [ ]:
def analyze_resume(resume_path, job_url):
    print("[1/4] Extracting resume...")
    resume_text = extract_resume_text(resume_path)
    
    print("[2/4] Fetching job description...")
    jd_text = fetch_website_contents(job_url)
    
    print("[3/4] Running LLM analysis (3 calls)...")
    score  = get_match_score(resume_text, jd_text)
    gaps   = get_gap_analysis(resume_text, jd_text)
    bullets = get_bullet_rewrites(resume_text, jd_text)
    
    print("[4/4] Streaming report...\n")
    stream_full_report(score, gaps, bullets, resume_text, jd_text)

analyze_resume(RESUME_PATH, JOB_URL)